# BacterioScope — Pipeline Walkthrough

This notebook walks through the complete BacterioScope analysis pipeline:

1. **Plate calibration** — detect the petri-dish circle and compute pixels-per-millimeter
2. **Disk detection** — locate antibiotic paper disks (HoughCircles or YOLOv8)
3. **Zone segmentation** — measure inhibition zone diameters (Otsu + watershed)
4. **CLSI classification** — map diameters to S / I / R using CLSI M100-Ed33 (2023)
5. **Evaluation** — compute clinical agreement metrics on a held-out set

**Prerequisites:** `pip install -e ".[all,dev]"`

In [ ]:
from __future__ import annotations

import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 6)})

## 1. Synthetic plate

We construct a synthetic Mueller-Hinton agar plate with six antibiotic disks and
realistic inhibition zone halos. In production, replace this with a photograph.

In [ ]:
SIZE = 540
CENTER = (SIZE // 2, SIZE // 2)
PLATE_R = 252
DISK_R = 14
PX_PER_MM_TRUE = SIZE / (90 * 1.14)  # ground-truth calibration

_ANGLES = [90, 30, 330, 270, 210, 150]
POSITIONS = [
    (CENTER[0] + int(170 * np.cos(np.radians(a))),
     CENTER[1] - int(170 * np.sin(np.radians(a))))
    for a in _ANGLES
]

# (antibiotic, zone_radius_px, expected_category)
DISKS = [
    ('Meropenem',     76, 'S'),
    ('Ciprofloxacin', 62, 'I'),
    ('Ampicillin',    36, 'R'),
    ('Ceftriaxone',   82, 'S'),
    ('Imipenem',      64, 'I'),
    ('Gentamicin',    44, 'S'),
]


def make_plate() -> np.ndarray:
    img = np.full((SIZE, SIZE, 3), [38, 38, 38], dtype=np.uint8)
    agar = np.full((SIZE, SIZE, 3), [150, 183, 205], dtype=np.uint8)

    mask = np.zeros((SIZE, SIZE), dtype=np.uint8)
    cv2.circle(mask, CENTER, PLATE_R, 255, -1)
    agar[mask > 0] = [136, 165, 186]

    for (cx, cy), (_, zr, _) in zip(POSITIONS, DISKS):
        zone = np.zeros((SIZE, SIZE), dtype=np.float32)
        cv2.circle(zone, (cx, cy), zr, 1.0, -1)
        zone = cv2.GaussianBlur(zone, (25, 25), 9)
        for c in range(3):
            agar[:, :, c] = np.clip(
                agar[:, :, c].astype(np.float32) + zone * 28, 0, 255
            ).astype(np.uint8)

    rng = np.random.default_rng(42)
    noise = rng.integers(-7, 8, (SIZE, SIZE, 3), dtype=np.int16)
    agar = np.clip(agar.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    img[mask > 0] = agar[mask > 0]

    for cx, cy in POSITIONS:
        cv2.circle(img, (cx, cy), DISK_R, (232, 232, 225), -1)
        cv2.circle(img, (cx, cy), DISK_R, (190, 190, 183), 1)

    cv2.circle(img, CENTER, PLATE_R, (88, 108, 118), 2)
    return img


plate = make_plate()
plt.imshow(cv2.cvtColor(plate, cv2.COLOR_BGR2RGB))
plt.title('Synthetic Mueller-Hinton plate')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Plate calibration

`calibration.py` detects the outer petri-dish circle with HoughCircles and computes
**pixels per millimeter** from the known plate diameter (90 mm standard).

In [ ]:
from bacterioscope.utils.calibration import calibrate_plate

px_per_mm, plate_diameter_px = calibrate_plate(plate, plate_diameter_mm=90.0)
print(f'px/mm : {px_per_mm:.3f}  (ground-truth: {PX_PER_MM_TRUE:.3f})')
print(f'Plate diameter detected: {plate_diameter_px:.0f} px')

## 3. Disk detection

`DiskDetector` tries YOLOv8 weights first; if none are present it falls back to
HoughCircles. The synthetic plate uses the Hough path.

In [ ]:
from bacterioscope.detection.detector import DiskDetector

detector = DiskDetector()
disks = detector.detect(plate)

vis = plate.copy()
for d in disks:
    cx, cy = int(d.center[0]), int(d.center[1])
    cv2.circle(vis, (cx, cy), int(d.radius) + 3, (220, 220, 50), 2)
    cv2.circle(vis, (cx, cy), 3, (220, 220, 50), -1)

plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f'Disk detection — {len(disks)} disk(s) found')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Detected {len(disks)} disk(s)')

## 4. Zone segmentation

`ZoneSegmenter` extracts a region of interest around each disk, applies Otsu
thresholding, and runs watershed to find the inhibition zone boundary.

In [ ]:
from bacterioscope.segmentation.watershed import ZoneSegmenter

segmenter = ZoneSegmenter()

# Use the known ground-truth positions so the walkthrough is deterministic
from bacterioscope.detection.detector import DiskResult

gt_disks = [
    DiskResult(center=(float(cx), float(cy)), radius=float(DISK_R), label='disk', confidence=1.0)
    for (cx, cy) in POSITIONS
]

zones = [segmenter.segment(plate, d, px_per_mm) for d in gt_disks]

vis2 = plate.copy()
colors = [(30, 180, 30), (20, 180, 220), (30, 30, 210),
          (30, 180, 30), (20, 180, 220), (30, 180, 30)]
for (cx, cy), z, clr in zip(POSITIONS, zones, colors):
    r_px = int(z.diameter_px / 2)
    cv2.circle(vis2, (cx, cy), r_px, clr, 2)
    cv2.putText(vis2, f'{z.diameter_mm:.1f}mm',
                (cx - 22, cy - r_px - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, clr, 1, cv2.LINE_AA)

plt.imshow(cv2.cvtColor(vis2, cv2.COLOR_BGR2RGB))
plt.title('Zone segmentation — measured diameters')
plt.axis('off')
plt.tight_layout()
plt.show()

## 5. CLSI 2023 classification

`CLSIClassifier` looks up each antibiotic in the embedded CLSI M100-Ed33 breakpoint
table and returns S (susceptible), I (intermediate), or R (resistant).

In [ ]:
from bacterioscope.classification.clsi import CLSIClassifier

classifier = CLSIClassifier()

CATEGORY_EMOJI = {'S': '[S]', 'I': '[I]', 'R': '[R]'}
COLORS_TERM = {'S': '\033[92m', 'I': '\033[93m', 'R': '\033[91m'}
RESET = '\033[0m'

print(f'{"Antibiotic":<16} {"Zone (mm)":>10} {"Expected":>10} {"Predicted":>10}')
print('-' * 52)
for (name, zr_px, expected), z in zip(DISKS, zones):
    result = classifier.classify(name, z.diameter_mm)
    predicted = result.category
    match = predicted == expected
    c = COLORS_TERM.get(predicted, '')
    mark = '  OK' if match else '  !'
    print(f'{name:<16} {z.diameter_mm:>10.1f} {expected:>10} {c}{predicted:>10}{RESET}{mark}')

## 6. Clinical agreement metrics

The `bacterioscope.evaluation` module computes ISO 20776-2 / FDA clinical
agreement metrics. Here we run a small deterministic example.

In [ ]:
from bacterioscope.evaluation import (
    categorical_agreement,
    collect_metrics,
    error_rates,
    essential_agreement,
    zone_diameter_stats,
)

# Collect predicted and reference categories from the walkthrough above
predicted_cats = []
reference_cats = []
predicted_mm = []
reference_mm = []

for (name, zr_px, expected), z in zip(DISKS, zones):
    result = classifier.classify(name, z.diameter_mm)
    predicted_cats.append(result.category)
    reference_cats.append(expected)
    predicted_mm.append(z.diameter_mm)
    # ground-truth diameter from known zone radius
    reference_mm.append(2 * zr_px / PX_PER_MM_TRUE)

ca = categorical_agreement(predicted_cats, reference_cats)
ea = essential_agreement(predicted_mm, reference_mm)
errors = error_rates(predicted_cats, reference_cats)
stats = zone_diameter_stats(predicted_mm, reference_mm)

print(f'Categorical Agreement (CA) : {ca * 100:.1f}%')
print(f'Essential Agreement  (EA)  : {ea * 100:.1f}%  (±2 mm diameter window)')
print(f'Very Major Errors   (VME)  : {errors["vme_count"]}  ({errors["vme_rate"] * 100:.1f}%)')
print(f'Major Errors        (ME)   : {errors["me_count"]}  ({errors["me_rate"] * 100:.1f}%)')
print(f'Minor Errors        (mE)   : {errors["minor_count"]}  ({errors["minor_rate"] * 100:.1f}%)')
print(f'MAE diameter               : {stats["mae_mm"]:.2f} mm')
print(f'Pearson r                  : {stats["pearson_r"]:.4f}')

## 7. Confusion matrix

In [ ]:
from bacterioscope.evaluation import sir_confusion_matrix

cm = sir_confusion_matrix(predicted_cats, reference_cats)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(['S', 'I', 'R'])
ax.set_yticklabels(['S', 'I', 'R'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Reference')
ax.set_title('S/I/R Confusion Matrix')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=13)
fig.colorbar(im)
plt.tight_layout()
plt.show()

## 8. Full metrics report

`collect_metrics` assembles all metrics into a single dict that `generate_markdown`
formats into a clinical-grade report.

In [ ]:
from bacterioscope.evaluation import collect_metrics
from bacterioscope.evaluation.report import generate_markdown

metrics = collect_metrics(
    predicted=predicted_cats,
    reference=reference_cats,
    measured_mm=predicted_mm,
    reference_mm=reference_mm,
)

print(generate_markdown(metrics))

## Next steps

- **Train YOLOv8**: `python -m bacterioscope.detection.train --data data/processed/`
- **Evaluate on dataset**: `python scripts/evaluate.py --csv data/processed/labels.csv`
- **Streamlit demo**: `streamlit run src/bacterioscope/app.py`
- **REST API**: `uvicorn bacterioscope.api.routes:app --reload`